Advanced Modeling — Ensembles, Tuning, and Full ML **Pipeline**

In [ ]:
#task 1 Decision tree classifier
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("cleaned_data.csv")

# Classification target
y = (df["Price"] > df["Price"].median()).astype(int)

# Features
X = df.drop("Price", axis=1)

# Convert categorical columns
X = pd.get_dummies(X, drop_first=True)

# Convert all columns to numeric
X = X.astype(float)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train Shape:", X_train_scaled.shape)
print("Test Shape :", X_test_scaled.shape)

In [ ]:
#Task 1 training and test accuracy
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train_scaled, y_train)

train_pred = dt.predict(X_train_scaled)
test_pred = dt.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("Training Accuracy:", round(train_acc, 4))
print("Test Accuracy    :", round(test_acc, 4))

In [ ]:
# task controlled decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Controlled Decision Tree
dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Train model
dt_controlled.fit(X_train_scaled, y_train)

# Predictions
train_pred = dt_controlled.predict(X_train_scaled)
test_pred = dt_controlled.predict(X_test_scaled)

# Accuracy
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("Controlled Decision Tree Results")
print("--------------------------------")
print("Training Accuracy:", round(train_acc, 4))
print("Test Accuracy    :", round(test_acc, 4))

In [ ]:
#Task 3  GENI vs Entropy
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# ==========================
# Gini Tree
# ==========================
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

gini_pred = gini_tree.predict(X_test_scaled)

gini_acc = accuracy_score(y_test, gini_pred)

# ==========================
# Entropy Tree
# ==========================
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

entropy_pred = entropy_tree.predict(X_test_scaled)

entropy_acc = accuracy_score(y_test, entropy_pred)

# ==========================
# Results Table
# ==========================
results = pd.DataFrame({
    "Criterion": ["Gini", "Entropy"],
    "Test Accuracy": [round(gini_acc, 4), round(entropy_acc, 4)]
})

print(results)

In [ ]:
#Task 4 Random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# ==========================
# Random Forest
# ==========================
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train model
rf.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = rf.predict(X_train_scaled)
y_test_pred = rf.predict(X_test_scaled)

# Probabilities for AUC
y_prob = rf.predict_proba(X_test_scaled)[:, 1]

# Metrics
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_prob)

print("Random Forest Results")
print("---------------------")
print("Training Accuracy :", round(train_acc, 4))
print("Test Accuracy     :", round(test_acc, 4))
print("ROC-AUC           :", round(auc, 4))

In [ ]:
#t4 retrieve mode feature
import pandas as pd

# Get feature importance scores
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

# Sort descending
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Top 5 features
top5 = importance_df.head(5)

print("Top 5 Important Features")
print(top5)
print("\nTop 5 Features by Importance\n")

for i, row in top5.iterrows():
    print(f"{row['Feature']} : {row['Importance']:.4f}")

In [ ]:
#Task4a gradient boosting
# gradient boosting classifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# ==========================
# Gradient Boosting Model
# ==========================
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train
gb.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = gb.predict(X_train_scaled)
y_test_pred = gb.predict(X_test_scaled)

# Probabilities for ROC-AUC
y_prob = gb.predict_proba(X_test_scaled)[:, 1]

# Metrics
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_prob)

print("Gradient Boosting Results")
print("-------------------------")
print("Training Accuracy :", round(train_acc, 4))
print("Test Accuracy     :", round(test_acc, 4))
print("ROC-AUC           :", round(auc, 4))

In [ ]:
# task 4 b advanced analysis
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ==============================
# 1. Find 5 least important features
# ==============================
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=True
)

lowest5 = importance_df.head(5)

print("5 Least Important Features")
print(lowest5)

low_features = lowest5["Feature"].tolist()

# ==============================
# 2. Remove those features
# ==============================
X_reduced = X.drop(columns=low_features)

# Create matching train/test sets
from sklearn.model_selection import train_test_split

X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale
from sklearn.preprocessing import StandardScaler

scaler_red = StandardScaler()

X_train_red_scaled = scaler_red.fit_transform(X_train_red)
X_test_red_scaled = scaler_red.transform(X_test_red)

# ==============================
# 3. Train Reduced Random Forest
# ==============================
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_red_scaled, y_train_red)

# ==============================
# 4. Compute AUC
# ==============================

# Full model AUC (already trained earlier)
full_prob = rf.predict_proba(X_test_scaled)[:, 1]
full_auc = roc_auc_score(y_test, full_prob)

# Reduced model AUC
reduced_prob = rf_reduced.predict_proba(X_test_red_scaled)[:, 1]
reduced_auc = roc_auc_score(y_test_red, reduced_prob)

print("\nAUC Comparison")
print("---------------------")
print("Full Model AUC    :", round(full_auc, 4))
print("Reduced Model AUC :", round(reduced_auc, 4))
print("Difference        :", round(full_auc - reduced_auc, 4))

In [ ]:
# 4b feature model
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# =====================================
# 1. Get feature importances
# =====================================
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

# Sort ascending (lowest first)
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=True
)

# Lowest 5 features
lowest5 = importance_df.head(5)

print("5 Lowest Importance Features")
print(lowest5)

# Feature names to remove
features_to_remove = lowest5["Feature"].tolist()

print("\nFeatures Removed:")
for f in features_to_remove:
    print(f)

# =====================================
# 2. Full Model AUC
# =====================================
full_prob = rf.predict_proba(X_test_scaled)[:, 1]

full_auc = roc_auc_score(y_test, full_prob)

# =====================================
# 3. Remove lowest-importance features
# =====================================
X_train_reduced = X_train.drop(
    columns=features_to_remove
)

X_test_reduced = X_test.drop(
    columns=features_to_remove
)

# =====================================
# 4. Train Reduced Random Forest
# =====================================
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(
    X_train_reduced,
    y_train
)

# =====================================
# 5. Reduced Model AUC
# =====================================
reduced_prob = rf_reduced.predict_proba(
    X_test_reduced
)[:, 1]

reduced_auc = roc_auc_score(
    y_test,
    reduced_prob
)

# =====================================
# 6. Results
# =====================================
print("\nFeature Ablation Results")
print("------------------------")
print("Full Model AUC    :", round(full_auc, 4))
print("Reduced Model AUC :", round(reduced_auc, 4))
print("Difference        :", round(reduced_auc - full_auc, 4))

In [ ]:
#task5 cross validation comparison
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import pandas as pd
import numpy as np

# 5-Fold Stratified Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

# Cross-validation
results = []

for name, model in models.items():

    auc_scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    results.append([
        name,
        round(np.mean(auc_scores), 4),
        round(np.std(auc_scores), 4)
    ])

# Results table
cv_results = pd.DataFrame(
    results,
    columns=["Model", "Mean AUC", "Std AUC"]
)

print(cv_results.sort_values("Mean AUC", ascending=False))

In [ ]:
#task 6 Hyper parameter tuning
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# ==========================
# Pipeline
# ==========================
rf_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# ==========================
# Parameter Grid
# ==========================
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

# ==========================
# Cross Validation
# ==========================
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ==========================
# Grid Search
# ==========================
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Fit Grid Search
grid_search.fit(X_train, y_train)

# ==========================
# Results
# ==========================
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated AUC:")
print(round(grid_search.best_score_, 4))

In [ ]:
#top combinations
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)

results = results.sort_values(
    by='mean_test_score',
    ascending=False
)

print(results[
    [
        'param_randomforestclassifier__n_estimators',
        'param_randomforestclassifier__max_depth',
        'param_randomforestclassifier__min_samples_leaf',
        'mean_test_score'
    ]
].head(10))

In [ ]:
#unscaledpipeline
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# Parameter Grid
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

# 5-Fold Stratified CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

# Fit on UNSCALED training data
grid_search.fit(X_train, y_train)

# Results
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated ROC-AUC:")
print(round(grid_search.best_score_, 4))

In [ ]:
#Task7 Manual learning curve
import pandas as pd
from sklearn.metrics import roc_auc_score
best_model = grid_search.best_estimator_
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for f in fractions:

    n_rows = int(f * len(X_train))

    # Take first n_rows
    X_sub = X_train.iloc[:n_rows]
    y_sub = y_train.iloc[:n_rows]

    # Train best pipeline
    best_model.fit(X_sub, y_sub)

    # Training AUC
    train_prob = best_model.predict_proba(X_sub)[:, 1]
    train_auc = roc_auc_score(y_sub, train_prob)

    # Test AUC
    test_prob = best_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_prob)

    results.append([
        f,
        round(train_auc, 4),
        round(test_auc, 4)
    ])

learning_curve_df = pd.DataFrame(
    results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

print(learning_curve_df)

In [ ]:
#Task 8 serealize best pipeline
import joblib

# Best model from GridSearchCV
best_pipeline = grid_search.best_estimator_

# Save model
joblib.dump(best_pipeline, 'best_model.pkl')

print("Model saved successfully!")

In [ ]:
# t 8 Short code block
import joblib
import pandas as pd

# Load saved model
model = joblib.load("best_model.pkl")

# Create two sample rows
sample_data = pd.DataFrame([
    X_train.iloc[0].to_dict(),
    X_train.iloc[1].to_dict()
])

# Predict
predictions = model.predict(sample_data)

print("Predictions:")
print(predictions)

In [ ]:
print(X_train.shape)
print(X_train.columns.tolist())

In [ ]:
# size of  saved model
import os

size_mb = os.path.getsize("best_model.pkl") / (1024 * 1024)
print(f"Model Size: {size_mb:.2f} MB")

In [ ]:
import pickle
import os
import joblib

model = joblib.load('best_model.pkl')

print(model)


In [ ]:
import os
import pickle
get_ipython().system

import joblib
print(os.path.exists("best_model.pkl"))

with open("best_model.pkl", "rb") as f:
    print(f.read(20))

model = joblib.load("best_model.pkl")

print(type(model))

print(model.named_steps)

joblib.dump(best_pipeline, "best_model_new.pkl")
best_pipeline = grid_search.best_estimator_
print("saved")
model = joblib.load("best_model.pkl")
